# Replacement and deletion ablation tests for enriched high-gap words

Ablation for these controlled elements: 

1. **replacement** with a manually defined low-gap control word
2. **deletion** of the selected enriched word

The main outcomes are **magnitude-based**:

- `abs_delta_valence`
- `abs_delta_arousal`
- `delta_affect_shift_mag = sqrt(delta_valence^2 + delta_arousal^2)`

In [1]:

from pathlib import Path
import ast
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, wilcoxon, ttest_rel, ttest_1samp

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## Paths and configuration

In [11]:
REPO_ROOT = Path.cwd()

# Try to use config if available; otherwise fall back to default folders.
processed_dir = REPO_ROOT / ".." / "data" / "processed"
tables_dir = REPO_ROOT / ".." / "reports" / "tables"
figures_dir = REPO_ROOT / ".." / "reports" / "figures"

try:
    import sys
    sys.path.insert(0, str(REPO_ROOT / ".." / "src"))
    from l2affect.utils.config import load_config, resolve  # type: ignore

    cfg = load_config(REPO_ROOT / ".." / "configs" / "config.yaml")
    processed_dir = resolve(cfg["paths"]["processed_dir"])
    tables_dir = resolve(cfg["paths"]["reports_tables_dir"])
    figures_dir = resolve(cfg["paths"]["reports_figures_dir"])
except Exception as e:
    print("Config not loaded (ok). Using default folders.")
    print("Reason:", e)

processed_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

output_dir = REPO_ROOT / "analysis_outputs"
data_path = processed_dir / "writestreak_full_with_predictions.csv"
word_path = processed_dir / "top30_highgap_words_mh_enriched_top5.csv"
gap_path = processed_dir / "1-imbault_gaps_va.csv"

ID_COL = "post_id"
TEXT_ID_COL = "text_id"
USER_COL = "user_id"
TEXT_COL = "text"
TOKENS_COL = "tokens"
N_TOKENS_COL = "n_tokens"
TIMESTAMP_COL = "created_at"

BASELINE_VALENCE_COL = "pred_valence_1_9"
BASELINE_AROUSAL_COL = "pred_arousal_1_9"

MASK_TOKEN = "[MASK]"
PREDICT_BATCH_SIZE = 64

# If True, only treated posts are rerun through the model.
RUN_ONLY_TREATED_POSTS = True

output_dir = REPO_ROOT / ".." / "data" / "analysis_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

print("Prediction file:", data_path)
print("Enriched word file:", word_path)


Prediction file: C:\Users\hus44\Code\Directed-Reading-Project\data\processed\writestreak_full_with_predictions.csv
Enriched word file: C:\Users\hus44\Code\Directed-Reading-Project\data\processed\top30_highgap_words_mh_enriched_top5.csv


In [12]:

posts = pd.read_csv(data_path)
enriched_df = pd.read_csv(word_path)
gap_df = pd.read_csv(gap_path) if gap_path is not None else None

print("Posts shape:", posts.shape)
print("Enriched word file shape:", enriched_df.shape)
if gap_df is not None:
    print("Gap file shape:", gap_df.shape)

display(posts.head(2))
display(enriched_df.head(10))
if gap_df is not None:
    display(gap_df.head(5))

Posts shape: (9168, 43)
Enriched word file shape: (30, 8)
Gap file shape: (2008, 8)


,post_id,post_fullname,author,user_id,created_utc,created_at,title,selftext,subreddit,permalink,score,num_comments,source_file,text,tokens,n_tokens,n_chars,post_index,days_since_first,user_post_count,high_gap_count,high_gap_density,high_gap_gapmag_mean,high_gap_gapmag_max,high_gap_valence_mean_signed,high_gap_arousal_mean_signed,swear_count,swear_density,swear_present,swear_unique_count,swear_types,high_gap_swear_count,mh_sim_max,mh_sim_mean_top3,mh_best_seed_idx,mh_best_seed,mh_best_seed_sim,mh_flag,text_id,pred_valence_raw,pred_arousal_raw,pred_valence_1_9,pred_arousal_1_9
0,j4p7n0,t3_j4p7n0,Stylelike,000a672e1864051f,1.601769e+09,2020-10-03 23:42:25+00:00,Streak 2: First aids,I think that learn first aids is indispensable...,WriteStreakEN,/r/WriteStreakEN/comments/j4p7n0/streak_2_firs...,1,4,202010.jsonl,Streak 2: First aids\n\nI think that learn fir...,"[""streak"", ""first"", ""aids"", ""i"", ""think"", ""tha...",65,350,2,1,4,1,0.015385,2.326908,2.326908,-1.61,1.68,0,0.0,0,0,[],0,0.141932,0.131127,12,I’ve had trouble concentrating on things like ...,0.141932,0,j4p7n0,-0.932517,0.294047,3.040777,4.359374
1,kk1866,t3_kk1866,Stylelike,000a672e1864051f,1.608912e+09,2020-12-25 16:01:26+00:00,Streak 1: Christmas and New Year.,"Firstly, I want to make it clear that I’m not ...",WriteStreakEN,/r/WriteStreakEN/comments/kk1866/streak_1_chri...,1,4,202012.jsonl,"Streak 1: Christmas and New Year.\n\nFirstly, ...","[""streak"", ""christmas"", ""and"", ""new"", ""year"", ...",115,639,4,84,4,1,0.008696,3.209190,3.209190,-0.50,3.17,0,0.0,0,0,[],0,0.354041,0.323948,1,Things I usually enjoy don’t feel enjoyable la...,0.354041,1,kk1866,2.007071,0.466640,6.832933,4.805484


,word,overall_token_count,overall_doc_count,overall_freq_per_10k_highgap_tokens,mh_token_count,mh_doc_count,mh_freq_per_10k_highgap_tokens,mh_enrichment_ratio
0,regain,9,9,4.548671,5.0,5.0,13.954786,3.067882
1,pace,66,61,33.356919,24.0,23.0,66.982975,2.008068
2,refresh,30,30,15.162236,10.0,10.0,27.909573,1.840729
3,assessment,15,14,7.581118,5.0,5.0,13.954786,1.840729
4,comfortable,254,236,128.373597,78.0,71.0,217.694669,1.695790
5,satisfaction,35,29,17.689275,10.0,10.0,27.909573,1.577768
6,breathe,53,48,26.786617,15.0,13.0,41.864359,1.562883
7,efficient,65,63,32.851511,18.0,17.0,50.237231,1.529221
8,normal,475,397,240.068735,129.0,108.0,360.033491,1.499710
9,extra,200,174,101.081573,54.0,43.0,150.711694,1.490991


,word,valence_l1,arousal_l1,valence_l2,arousal_l2,gap_valence,gap_arousal,gap_mag
0,abandon,2.84,3.73,2.43,3.58,-0.41,-0.15,0.436578
1,abdomen,5.43,3.68,4.88,4.62,-0.55,0.94,1.089082
2,abduction,2.05,5.33,2.91,5.39,0.86,0.06,0.862090
3,ability,7.00,4.85,7.33,5.29,0.33,0.44,0.550000
4,absence,3.86,4.30,3.42,3.72,-0.44,-0.58,0.728011


In [13]:

required_post_cols = [
    ID_COL, TEXT_ID_COL, USER_COL, TEXT_COL, TOKENS_COL, TIMESTAMP_COL, N_TOKENS_COL,
    BASELINE_VALENCE_COL, BASELINE_AROUSAL_COL
]
missing_post_cols = [c for c in required_post_cols if c not in posts.columns]
if missing_post_cols:
    raise ValueError(f"Prediction file is missing required columns: {missing_post_cols}")

if "word" not in enriched_df.columns:
    raise ValueError("Enriched word file must contain a 'word' column.")

## Manual low-gap replacement dictionary

This dictionary maps each selected enriched high-gap word to a manually chosen **lower-gap** replacement word that is intended to preserve rough grammatical category and approximate English meaning.

In [14]:

replacement_dict = {
    "regain": "recover",
    "pace": "rhythm",
    "refresh": "update",
    "assessment": "measure",
    "comfortable": "relaxed",
    "satisfaction": "ease",
    "breathe": "inhale",
    "efficient": "helpful",
    "normal": "typical",
    "extra": "spare",
    "assignment": "homework",
    "good": "acceptable",
    "inspiration": "influence",
    "decide": "determine",
    "precious": "rare",
    "fresh": "crisp",
    "grocery": "supermarket",
    "external": "foreign",
    "ideal": "suitable",
    "option": "route",
    "picnic": "activity",
    "special": "distinct",
    "wonderful": "delightful",
    "begin": "launch",
    "thousand": "million",
    "great": "important",
    "graduate": "degree",
    "lucky": "happy",
    "beginning": "introduction",
    "pet": "animal",
}

In [15]:

selected_words = (
    enriched_df["word"]
    .astype(str)
    .str.strip()
    .str.lower()
    .dropna()
    .unique()
)

selected_word_set = set(selected_words)

missing_map_keys = sorted(selected_word_set - set(replacement_dict.keys()))
extra_map_keys = sorted(set(replacement_dict.keys()) - selected_word_set)

print("Selected enriched words:", len(selected_word_set))
print("Replacement entries:", len(replacement_dict))
print("Missing replacement entries:", missing_map_keys)
print("Extra replacement entries:", extra_map_keys)

if missing_map_keys:
    raise ValueError("Replacement dictionary is missing entries for some selected words.")

Selected enriched words: 30
Replacement entries: 30
Missing replacement entries: []
Extra replacement entries: []


Validation against the attached gap lexicon

In [16]:

if gap_df is not None:
    gap_work = gap_df.copy()
    gap_work["word"] = gap_work["word"].astype(str).str.strip().str.lower()
    gap_lookup = dict(zip(gap_work["word"], pd.to_numeric(gap_work["gap_mag"], errors="coerce")))

    replacement_check = pd.DataFrame({
        "original_word": sorted(selected_word_set),
        "replacement_word": [replacement_dict[w] for w in sorted(selected_word_set)],
    })
    replacement_check["original_gap_mag"] = replacement_check["original_word"].map(gap_lookup)
    replacement_check["replacement_gap_mag"] = replacement_check["replacement_word"].map(gap_lookup)
    replacement_check["gap_reduction"] = (
        replacement_check["original_gap_mag"] - replacement_check["replacement_gap_mag"]
    )
    replacement_check["replacement_found_in_gap_lexicon"] = replacement_check["replacement_gap_mag"].notna().astype(int)

    display(replacement_check.sort_values("gap_reduction", ascending=False))
else:
    print("No gap lexicon found, skipping gap validation.")

,original_word,replacement_word,original_gap_mag,replacement_gap_mag,gap_reduction,replacement_found_in_gap_lexicon
9,extra,spare,2.969512,0.180278,2.789234,1
11,good,acceptable,2.874456,0.346699,2.527758,1
0,assessment,measure,2.696145,0.241868,2.454278,1
22,picnic,activity,2.673013,0.528110,2.144903,1
29,wonderful,delightful,2.334202,0.202485,2.131718,1
27,special,distinct,3.209190,1.167305,2.041885,1
3,beginning,introduction,2.760652,0.790759,1.969893,1
6,decide,determine,2.370021,0.411096,1.958925,1
13,great,important,2.721176,0.800062,1.921114,1
8,external,foreign,2.293687,0.522015,1.771672,1


## Rebuild treatment variables from the enriched word list

In [17]:

def parse_tokens(x):
    if isinstance(x, list):
        return [str(t).strip().lower() for t in x]
    if pd.isna(x):
        return []
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                return [str(t).strip().lower() for t in parsed]
        except Exception:
            pass
    return []

analysis_df = posts.copy()
analysis_df["tokens_list"] = analysis_df[TOKENS_COL].apply(parse_tokens)

analysis_df["selected_hits"] = analysis_df["tokens_list"].apply(
    lambda toks: [t for t in toks if t in selected_word_set]
)
analysis_df["selected_hit_count"] = analysis_df["selected_hits"].apply(len)
analysis_df["selected_hit_type_count"] = analysis_df["selected_hits"].apply(lambda xs: len(set(xs)))
analysis_df["selected_hit_prop"] = (
    analysis_df["selected_hit_count"] / analysis_df[N_TOKENS_COL].clip(lower=1)
)
analysis_df["has_selected_high_gap_word"] = (analysis_df["selected_hit_count"] > 0).astype(int)
analysis_df["selected_hits_str"] = analysis_df["selected_hits"].apply(lambda xs: ", ".join(sorted(set(xs))))

sanity = pd.DataFrame({
    "metric": [
        "total_posts",
        "posts_with_selected_high_gap_word",
        "posts_without_selected_high_gap_word",
        "share_with_selected_high_gap_word",
    ],
    "value": [
        len(analysis_df),
        int(analysis_df["has_selected_high_gap_word"].sum()),
        int((analysis_df["has_selected_high_gap_word"] == 0).sum()),
        float(analysis_df["has_selected_high_gap_word"].mean()),
    ]
})

display(sanity)

,metric,value
0,total_posts,9168.000000
1,posts_with_selected_high_gap_word,6482.000000
2,posts_without_selected_high_gap_word,2686.000000
3,share_with_selected_high_gap_word,0.707024


## Build ablation functions

Two conditions are created:

- `replacement_lowgap`: replace selected enriched words with manually matched low-gap alternatives
- `deletion`: remove selected enriched words entirely

In [18]:

escaped_words = sorted([re.escape(w) for w in selected_word_set], key=len, reverse=True)
word_pattern = re.compile(r"\b(" + "|".join(escaped_words) + r")\b", flags=re.IGNORECASE)

def preserve_case(source_word, replacement_word):
    if source_word.isupper():
        return replacement_word.upper()
    if source_word.istitle():
        return replacement_word.title()
    return replacement_word

def clean_text_after_deletion(text):
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    return text.strip()

def replace_selected_words(text, replacement_dict=replacement_dict):
    if pd.isna(text):
        return ""
    text = str(text)

    def repl(match):
        original = match.group(0)
        key = original.lower()
        repl_word = replacement_dict.get(key, original)
        return preserve_case(original, repl_word)

    out = word_pattern.sub(repl, text)
    return clean_text_after_deletion(out)

def delete_selected_words(text):
    if pd.isna(text):
        return ""
    text = str(text)
    out = word_pattern.sub("", text)
    return clean_text_after_deletion(out)

analysis_df["text_replacement_lowgap"] = analysis_df[TEXT_COL].apply(replace_selected_words)
analysis_df["text_deletion"] = analysis_df[TEXT_COL].apply(delete_selected_words)

analysis_df["replacement_changed_text"] = (
    analysis_df["text_replacement_lowgap"] != analysis_df[TEXT_COL].astype(str)
).astype(int)
analysis_df["deletion_changed_text"] = (
    analysis_df["text_deletion"] != analysis_df[TEXT_COL].astype(str)
).astype(int)

display(
    analysis_df.loc[
        analysis_df["has_selected_high_gap_word"] == 1,
        [ID_COL, TEXT_COL, "text_replacement_lowgap", "text_deletion", "selected_hits_str"]
    ].head(3)
)

,post_id,text,text_replacement_lowgap,text_deletion,selected_hits_str
1,kk1866,"Streak 1: Christmas and New Year.\n\nFirstly, ...","Streak 1: Christmas and New Year. Firstly, I w...","Streak 1: Christmas and New Year. Firstly, I w...",special
3,18sthvq,Streak 2: The start of a story (fiction). note...,Streak 2: The start of a story (fiction). note...,Streak 2: The start of a story (fiction). note...,fresh
4,18tq0cp,Streak X\n\nI wish you a good weekend 😁,Streak X I wish you a acceptable weekend 😁,Streak X I wish you a weekend 😁,good


In [19]:

changed_summary = pd.DataFrame({
    "perturbation": ["replacement_lowgap", "deletion"],
    "treated_posts": [
        int(analysis_df["has_selected_high_gap_word"].sum()),
        int(analysis_df["has_selected_high_gap_word"].sum()),
    ],
    "changed_treated_posts": [
        int(((analysis_df["has_selected_high_gap_word"] == 1) & (analysis_df["replacement_changed_text"] == 1)).sum()),
        int(((analysis_df["has_selected_high_gap_word"] == 1) & (analysis_df["deletion_changed_text"] == 1)).sum()),
    ],
})
display(changed_summary)

,perturbation,treated_posts,changed_treated_posts
0,replacement_lowgap,6482,6482
1,deletion,6482,6482


In [21]:
base_cols = [
    ID_COL, TEXT_ID_COL, USER_COL, TIMESTAMP_COL,
    TEXT_COL, BASELINE_VALENCE_COL, BASELINE_AROUSAL_COL,
    "selected_hit_count", "selected_hit_type_count", "selected_hit_prop",
    "has_selected_high_gap_word", "selected_hits_str"
]

work_df = analysis_df.copy()
if RUN_ONLY_TREATED_POSTS:
    work_df = work_df.loc[work_df["has_selected_high_gap_word"] == 1].copy()

# -----------------------------
# Replacement condition
# -----------------------------
replacement_df = work_df[base_cols].copy()
replacement_df["original_post_id"] = replacement_df[ID_COL].astype(str)
replacement_df[ID_COL] = replacement_df[ID_COL].astype(str) + "__replace_lowgap"
replacement_df["perturbation"] = "replacement_lowgap"
replacement_df["text_counterfactual"] = work_df["text_replacement_lowgap"].astype(str)
replacement_df["text"] = replacement_df["text_counterfactual"]

# -----------------------------
# Deletion condition
# -----------------------------
deletion_df = work_df[base_cols].copy()
deletion_df["original_post_id"] = deletion_df[ID_COL].astype(str)
deletion_df[ID_COL] = deletion_df[ID_COL].astype(str) + "__delete"
deletion_df["perturbation"] = "deletion"
deletion_df["text_counterfactual"] = work_df["text_deletion"].astype(str)
deletion_df["text"] = deletion_df["text_counterfactual"]

# -----------------------------
# Save as two separate CSVs
# -----------------------------
replacement_out = processed_dir / "9_replacement_data.csv"
deletion_out = processed_dir / "9_deletion_data.csv"

replacement_df.to_csv(replacement_out, index=False)
deletion_df.to_csv(deletion_out, index=False)

display(replacement_df.head())
display(deletion_df.head())

print("Replacement rows:", len(replacement_df))
print("Deletion rows:", len(deletion_df))
print("Saved:", replacement_out.resolve())
print("Saved:", deletion_out.resolve())

,post_id,text_id,user_id,created_at,text,pred_valence_1_9,pred_arousal_1_9,selected_hit_count,selected_hit_type_count,selected_hit_prop,has_selected_high_gap_word,selected_hits_str,original_post_id,perturbation,text_counterfactual
1,kk1866__replace_lowgap,kk1866,000a672e1864051f,2020-12-25 16:01:26+00:00,"Streak 1: Christmas and New Year. Firstly, I w...",6.832933,4.805484,1,1,0.008696,1,special,kk1866,replacement_lowgap,"Streak 1: Christmas and New Year. Firstly, I w..."
3,18sthvq__replace_lowgap,18sthvq,002305cde36be93b,2023-12-28 13:06:30+00:00,Streak 2: The start of a story (fiction). note...,5.790307,8.743247,1,1,0.002513,1,fresh,18sthvq,replacement_lowgap,Streak 2: The start of a story (fiction). note...
4,18tq0cp__replace_lowgap,18tq0cp,002305cde36be93b,2023-12-29 15:44:48+00:00,Streak X I wish you a acceptable weekend 😁,4.755822,6.276048,1,1,0.125000,1,good,18tq0cp,replacement_lowgap,Streak X I wish you a acceptable weekend 😁
5,18vcv2t__replace_lowgap,18vcv2t,002305cde36be93b,2023-12-31 17:42:39+00:00,Streak 4: 'carbid schieten' For today's streak...,5.946425,4.145765,1,1,0.005917,1,normal,18vcv2t,replacement_lowgap,Streak 4: 'carbid schieten' For today's streak...
7,18xqnzc__replace_lowgap,18xqnzc,002305cde36be93b,2024-01-03 18:49:33+00:00,Streak 7: failing or a learning opportunity It...,3.756989,6.613257,1,1,0.015873,1,option,18xqnzc,replacement_lowgap,Streak 7: failing or a learning opportunity It...


,post_id,text_id,user_id,created_at,text,pred_valence_1_9,pred_arousal_1_9,selected_hit_count,selected_hit_type_count,selected_hit_prop,has_selected_high_gap_word,selected_hits_str,original_post_id,perturbation,text_counterfactual
1,kk1866__delete,kk1866,000a672e1864051f,2020-12-25 16:01:26+00:00,"Streak 1: Christmas and New Year. Firstly, I w...",6.832933,4.805484,1,1,0.008696,1,special,kk1866,deletion,"Streak 1: Christmas and New Year. Firstly, I w..."
3,18sthvq__delete,18sthvq,002305cde36be93b,2023-12-28 13:06:30+00:00,Streak 2: The start of a story (fiction). note...,5.790307,8.743247,1,1,0.002513,1,fresh,18sthvq,deletion,Streak 2: The start of a story (fiction). note...
4,18tq0cp__delete,18tq0cp,002305cde36be93b,2023-12-29 15:44:48+00:00,Streak X I wish you a weekend 😁,4.755822,6.276048,1,1,0.125000,1,good,18tq0cp,deletion,Streak X I wish you a weekend 😁
5,18vcv2t__delete,18vcv2t,002305cde36be93b,2023-12-31 17:42:39+00:00,Streak 4: 'carbid schieten' For today's streak...,5.946425,4.145765,1,1,0.005917,1,normal,18vcv2t,deletion,Streak 4: 'carbid schieten' For today's streak...
7,18xqnzc__delete,18xqnzc,002305cde36be93b,2024-01-03 18:49:33+00:00,Streak 7: failing or a learning opportunity It...,3.756989,6.613257,1,1,0.015873,1,option,18xqnzc,deletion,Streak 7: failing or a learning opportunity It...


Replacement rows: 6482
Deletion rows: 6482
Saved: C:\Users\hus44\Code\Directed-Reading-Project\data\processed\9_replacement_data.csv
Saved: C:\Users\hus44\Code\Directed-Reading-Project\data\processed\9_deletion_data.csv


## Export CSV for the SemEval notebook

In [ ]:

counterfactual_df.to_csv(COUNTERFACTUAL_INPUT_CSV, index=False)
print("Saved counterfactual inference input to:", COUNTERFACTUAL_INPUT_CSV.resolve())

inference_ready_preview = counterfactual_df[
    [USER_COL, ID_COL, "original_post_id", TIMESTAMP_COL, "perturbation", "text"]
].head()

display(inference_ready_preview)